<a href="https://colab.research.google.com/github/chinh-hoangduc/QuantEcon-exercises/blob/main/lecture97_rep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install jax

In [3]:
from typing import NamedTuple
import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp
import jax.scipy as jsp
import jax

The first step in solving is to create the objects for households, firms, and prices in the economy.

In [4]:
class Household(NamedTuple):
    J: int
    j_grid: jnp.ndarray
    a_grid: jnp.ndarray
    γ_grid: jnp.ndarray
    Π: jnp.ndarray
    l_grid: jnp.ndarray
    β: float
    μ0: jnp.ndarray
    V_term: jnp.ndarray

def create_household(
    J: int = 50,
    a_min: float = 0,
    a_max: float = 10,
    a_num: int = 200,
    γ_grid = [0.5, 1.5],
    Π = [[0.9, 0.1], [0.1, 0.9]],
    β: float = 0.96
) -> Household:

    # create state grids
    a_grid = jnp.linspace(a_min, a_max, a_num)
    γ_grid, Π = map(jnp.asarray, (γ_grid, Π))

    j_grid = jnp.arange(J)

    # deterministic productivity age-profile
    l1, l2, l3 = 0.5, 0.05, -0.0008
    def l(j):
        return l1 + l2 * j + l3 * j ** 2
    l_grid = l(j_grid)

    # initial distribution: 0 asset and equal probability of \gamma
    μ0 = jnp.zeros((a_num, γ_grid.size))
    μ0 = μ0.at[0, :].set(1/γ_grid.size).reshape((a_grid.size*γ_grid.size))

    # set up terminal V with shape = ((a*γ),) to be used in DSDP later
    V_term = jnp.zeros((a_num * γ_grid.size))

    return Household(J, j_grid, a_grid, γ_grid, Π, l_grid, β, μ0, V_term)

In [6]:
household = create_household()

In [7]:
class Firm(NamedTuple):
    Z: float
    α: float

def create_firm(
    Z: float = 1,
    α: float = 0.3
) -> Firm:
    return Firm(Z, α)

In [8]:
firm = create_firm()

In [9]:
class Price(NamedTuple):
    r: float
    w: float

def create_price(
    r: float = 0.05,   # intitial guess/ place holder
    w: float = 1   # intitial guess/ place holder
) -> Price:
    return Price(r, w)

In [10]:
price = create_price()

Then write down the specifications of some functions.

In [11]:
def u(c: float, ν: float = 0.5):
   return c ** (1 - ν) / (1 - ν)

def interest(K: float, L: float, firm: Firm):
    return firm.α * firm.Z * (K / L)**(firm.α - 1)

def wage(K: float, L: float, firm: Firm):
    return (1 - firm.α) * firm.Z * (K / L)**(firm.α)

In [12]:
def find_τ(policy, price: Price, aggs):
    D, D_next, G, δ_grid = policy
    r, w = price
    K, L = aggs

    return (D - D_next + r*D + G - jnp.sum(δ_grid)) / (w*L + r*(D + K))

Next, we solve for the policy of households using backward induction.

We do that by the DSDP approach.

That requires defining the return function `R` and the transition matrix `Q`.

The matrix `Q` will take the form `Q(state, action, next-state)`, which will be fixed through iterations.

The return `R` will take in a loop value of age-`j`, the index of iteration.

In [13]:
def create_Q(household: Household):

    J, j_grid, a_grid, γ_grid, Π, l_grid, β, μ0, V_term = household

    num_state = a_grid.size * γ_grid.size
    num_action = a_grid.size

    Q1 = jsp.linalg.block_diag(*[Π.T]*num_action)
    Q2 = Q1.reshape((num_state, num_action, γ_grid.size))
    Q3 = jnp.tile(Q2, (1, 1, num_action))
    Q = Q3.T

    return Q

def create_R(
    j: int,
    household: Household,
    price: Price,
    τ: float,
    δ_grid: jnp.ndarray
):
    J, j_grid, a_grid, γ_grid, Π, l_grid, β, μ0, V_term = household
    r, w = price

    num_state = a_grid.size * γ_grid.size
    num_action = a_grid.size

    a = a_grid[:, None, None]
    γ = γ_grid[None, :, None]
    ap = a_grid[None, None, :]
    c = (1 + r*(1 - τ))*a + (1 - τ)*w*l_grid[j]*γ - δ_grid[j] - ap

    R = jnp.where(c > 0, u(c), -jnp.inf)
    R = R.reshape((num_state, num_action))

    return R

In [14]:
Q = create_Q(household)

Then the backward induction operator will wrap around `Q` and `R`.

Two things to note:
1. the value function is `V(state, )` not the usual `V(a, γ)`
2. for this inner loop, we will take prices and policies as given.

The operator used inside `jax.lax.scan` requires `(carry, loop-index)` and returns `(carry, ouputs)`.

In [15]:
def backward_vfi(
    household: Household,
    price: Price,
    Q,
    τ, δ_grid
):
    J, j_grid, a_grid, γ_grid, Π, l_grid, β, μ0, V_term = household

    def operator(V_next, j):   # `carry` goes first, then the X index
        Rj = create_R(j, household, price, τ, δ_grid)
        vals = Rj + β*Q.dot(V_next)
        σj_idx = jnp.argmax(vals, axis=1)
        Vj = jnp.max(vals, axis=1)
        return Vj, (Vj, σj_idx)

    init_V = V_term   # the initial carry
    js = jnp.arange(J-1, -1, -1)   # the loop index

    _, outputs = jax.lax.scan(operator, init = init_V, xs = js)
    V, σ_idx = outputs
    V = V[::-1]
    σ_idx = σ_idx[::-1]

    return V, σ_idx

For running one inner loop, assume some values of policies.

In [16]:
τ, δ_grid = 0.15, jnp.zeros(household.J)

In [17]:
V, σ_idx = backward_vfi(household, price, Q, τ, δ_grid)

Then the next step is to calculate the distribution from the optimal policy, which will then be used for calculating aggregates.

Note that the policy selects optimal asset directly on the grid points of current asset.

Therefore, we can use `Q` as the main workhorse.

In [18]:
def pop_dist(Q, σ_idx, household: Household):

    J, j_grid, a_grid, γ_grid, Π, l_grid, β, μ0, V_term = household
    num_state = a_grid.size * γ_grid.size

    def update_pop_j(μ_j, j):
        Qσ = Q[jnp.arange(num_state), σ_idx[j]]
        μ_next = μ_j @ Qσ

        return μ_next, μ_next

    init_μ = μ0   # NOTE: shape is (age, state)
    js = jnp.arange(J-1)

    _, μ = jax.lax.scan(update_pop_j, init_μ, js)

    # this mu is only from age 1 to J, so we have to concatenate the initial distribution
    μ = jnp.concatenate([init_μ[None, :], μ], axis=0)

    return μ

In [39]:
μ = pop_dist(Q, σ_idx, household)

Then we can calculate the aggregate capital and labour using this distribution.

In [32]:
def calculate_aggs(μ, household: Household):

    J, j_grid, a_grid, γ_grid, Π, l_grid, β, μ0, V_term = household

    μ = jnp.reshape(μ, (J, a_grid.size, γ_grid.size))

    K = jnp.sum(μ * a_grid[None, :, None]) / J

    L = jnp.sum(l_grid[:, None, None] * μ * γ_grid[None, None, :]) / J

    return K, L

In [42]:
K, L = calculate_aggs(μ, household)

Array(1.0781993, dtype=float32)

Then we can see if the market prices line up with our initial prediction.

In [45]:
r = interest(K, L, firm)
w = wage(K, L, firm)

Array(0.8243317, dtype=float32)

We can then put all these into a loop to calculate the steady-state equilibrium.

In [89]:
def calculate_ss(policy_target, firm, household, Q, tol=1e-6):

    D, G, δ_grid = policy_target

    def cond_fn(loop_state):
        D, G, δ_grid, τ, r, w, K, L, r_new, w_new = loop_state
        error = (r_new - r)**2/r + (w_new - w)**2/w
        return (error > tol)

    def body_fn(loop_state):
        D, G, δ_grid, τ, r, w, K, L, r_new, w_new = loop_state
        r = r_new
        w = w_new
        price = create_price(r=r, w=w)

        V, σ_idx = backward_vfi(household, price, Q, τ, δ_grid)
        μ = pop_dist(Q, σ_idx, household)
        K, L = calculate_aggs(μ, household)

        D_next = D
        τ = find_τ([D, D_next, G, δ_grid], price, [K, L])

        r_new = (r + interest(K, L, firm)) / 2
        w_new = (w + wage(K, L, firm)) / 2
        return D, G, δ_grid, τ, r, w, K, L, r_new, w_new

    τ_ini = 0.15
    K_ini = 3.
    L_ini = 1.
    r_ini = 0.05
    w_ini = 1.
    initial_state = (D, G, δ_grid, τ_ini, r_ini, w_ini, K_ini, L_ini, r_ini-1, w_ini-1)

    D, G, δ_grid, τ, r, w, K, L, r_new, w_new = jax.lax.while_loop(cond_fn, body_fn, initial_state)

    return r, w, K, L

In [86]:
find_τ([0, 0, 0.1, np.zeros(household.j_grid.size)], price, [0.0001, 0.0001])

Array(952.381, dtype=float32)

In [90]:
policy_target = [0, 0.1, np.zeros(household.j_grid.size)]

r, w, K, L = calculate_ss(policy_target, firm, household, Q, tol=1e-4)

In [92]:
print(r, w, K, L)

0.08412562 1.2224923 6.627401 1.0781993
